# Vilier Colab Runner

Notebook nay mount Google Drive, clone/pull repo, cai dependencies, chay `bash run.sh` voi input lay tu Drive va output ghi lai Drive.

In [ ]:
# Sua cac gia tri nay truoc khi chay neu can.
REPO_URL = "https://github.com/ngocbao220/vilier.git"
BRANCH = "main"
PROJECT_DIR = "/content/vilier"

# Dat file audio trong Google Drive, vi du: MyDrive/vilier/input/real.wav
DRIVE_AUDIO_PATH = "/content/drive/MyDrive/VDT-TurnTaking/inputs/real.wav"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/VDT-TurnTaking/outputs"

# Bat ASR neu muon chay PhoWhisper tren GPU Colab. Mac dinh giu false de tach speaker truoc.
ENABLE_ASR = False
ENABLE_STATE_LABELING = False
DRY_RUN = False


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import subprocess
from pathlib import Path

project_dir = Path(PROJECT_DIR)
if project_dir.exists():
    subprocess.run(["git", "-C", str(project_dir), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(project_dir), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(project_dir), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(project_dir)], check=True)

os.chdir(project_dir)
print("Repo:", project_dir)


In [ ]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


In [9]:
import json
import os
import sys
from pathlib import Path

try:
    from google.colab import userdata
except Exception:
    userdata = None

def set_secret_from_colab(secret_name, env_name):
    if os.environ.get(env_name) or userdata is None:
        return
    try:
        value = userdata.get(secret_name)
    except Exception:
        value = None
    if value:
        os.environ[env_name] = value

set_secret_from_colab("HUGGINGFACE_TOKEN", "HUGGINGFACE_TOKEN")
set_secret_from_colab("HF_TOKEN", "HF_TOKEN")
set_secret_from_colab("DASHSCOPE_API_KEY", "DASHSCOPE_API_KEY")

audio_path = Path(DRIVE_AUDIO_PATH)
output_dir = Path(DRIVE_OUTPUT_DIR)
if not audio_path.exists():
    raise FileNotFoundError(f"Drive audio not found: {audio_path}")
output_dir.mkdir(parents=True, exist_ok=True)

with open("config.json", "r", encoding="utf-8") as handle:
    config = json.load(handle)

config.setdefault("entrypoint", {})["input_path"] = str(audio_path)
config.setdefault("entrypoint", {})["output_path"] = str(output_dir)
config.setdefault("runtime", {})["dry_run"] = bool(DRY_RUN)
config.setdefault("diarization", {})["device"] = "cuda"
config.setdefault("asr", {})["enabled"] = bool(ENABLE_ASR)
config.setdefault("asr", {})["device"] = 0 if ENABLE_ASR else "cpu"
config.setdefault("state_labeling", {})["enabled"] = bool(ENABLE_STATE_LABELING)

config_path = Path("config.colab.json")
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")

os.environ["CONFIG_PATH"] = str(config_path)
os.environ["INPUT_PATH"] = str(audio_path)
os.environ["OUTPUT_PATH"] = str(output_dir)
os.environ["PYTHON_BIN"] = sys.executable
if DRY_RUN:
    os.environ["DRY_RUN"] = "1"

print("CONFIG_PATH=", os.environ["CONFIG_PATH"])
print("INPUT_PATH=", os.environ["INPUT_PATH"])
print("OUTPUT_PATH=", os.environ["OUTPUT_PATH"])
print("ENABLE_ASR=", ENABLE_ASR)
print("ENABLE_STATE_LABELING=", ENABLE_STATE_LABELING)


CONFIG_PATH= config.colab.json
INPUT_PATH= /content/drive/MyDrive/VDT-TurnTaking/inputs/real.wav
OUTPUT_PATH= /content/drive/MyDrive/VDT-TurnTaking/outputs
ENABLE_ASR= False
ENABLE_STATE_LABELING= False


In [10]:
!bash run.sh


[INFO] Pipeline component usage
| Component          | Enabled | Backend          | Model                              |
| ------------------ | ------- | ---------------- | ---------------------------------- |
| VAD                | V       | silero           | silero_vad                         |
| Diarization        | V       | pixit            | pyannote/speech-separation-ami-1.0 |
| Music separation   | X       | demucs           | htdemucs                           |
| Overlap separation | X       | sepreformer      | SepReformer_Base_WSJ0              |
| ASR                | X       | phowhisper_local | vinai/PhoWhisper-large             |
| State labeling     | X       | qwen             | qwen3.8-max                        |
[INFO] Vilier pipeline
└── batch
    ├── input_path=../drive/MyDrive/VDT-TurnTaking/inputs/real.wav
    ├── output_path=../drive/MyDrive/VDT-TurnTaking/outputs
    ├── log_dir=logs/2026-08-26
    ├── state_dir=../drive/MyDrive/VDT-TurnTaking/outputs/real/s

In [11]:
from pathlib import Path

audio_id = Path(DRIVE_AUDIO_PATH).stem
result_dir = Path(DRIVE_OUTPUT_DIR) / audio_id
print("Result dir:", result_dir)
for path in sorted(result_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(result_dir))


Result dir: /content/drive/MyDrive/VDT-TurnTaking/outputs/real
audio.standardized.wav
diarization_chunks/chunk_1.wav
diarization_chunks/chunk_2.wav
vad.json
vad.txt
vad_audio/audio_1.wav
vad_audio/audio_10.wav
vad_audio/audio_11.wav
vad_audio/audio_12.wav
vad_audio/audio_13.wav
vad_audio/audio_14.wav
vad_audio/audio_15.wav
vad_audio/audio_16.wav
vad_audio/audio_17.wav
vad_audio/audio_18.wav
vad_audio/audio_19.wav
vad_audio/audio_2.wav
vad_audio/audio_20.wav
vad_audio/audio_21.wav
vad_audio/audio_22.wav
vad_audio/audio_23.wav
vad_audio/audio_24.wav
vad_audio/audio_25.wav
vad_audio/audio_26.wav
vad_audio/audio_27.wav
vad_audio/audio_28.wav
vad_audio/audio_29.wav
vad_audio/audio_3.wav
vad_audio/audio_30.wav
vad_audio/audio_31.wav
vad_audio/audio_32.wav
vad_audio/audio_33.wav
vad_audio/audio_34.wav
vad_audio/audio_35.wav
vad_audio/audio_36.wav
vad_audio/audio_37.wav
vad_audio/audio_38.wav
vad_audio/audio_39.wav
vad_audio/audio_4.wav
vad_audio/audio_40.wav
vad_audio/audio_41.wav
vad_audio/a